In [1]:
import os
import pandas as pd
from pathlib import Path

current_dir = Path(os.getcwd())
DATA_PATH = current_dir.parent / "data" 



**Q1. Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu?** *(Tính từ `orders.csv`)*

- [ ] A) 30 ngày
- [ ] B) 90 ngày
- **C) 144 ngày**
- [ ] D) 365 ngày

In [2]:
order_df = pd.read_csv(DATA_PATH / "raw" / "orders.csv")

In [3]:
order_df.head()

,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source
0,1,2012-07-04,58578,1109,delivered,credit_card,desktop,paid_search
1,2,2012-07-04,58621,1330,returned,cod,mobile,paid_search
2,3,2012-07-04,58811,1473,delivered,credit_card,desktop,direct
3,4,2012-07-04,59453,2360,delivered,credit_card,desktop,referral
4,6,2012-07-06,57821,2886,delivered,paypal,mobile,email_campaign


In [4]:
order_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype
---  ------          --------------   -----
 0   order_id        646945 non-null  int64
 1   order_date      646945 non-null  str  
 2   customer_id     646945 non-null  int64
 3   zip             646945 non-null  int64
 4   order_status    646945 non-null  str  
 5   payment_method  646945 non-null  str  
 6   device_type     646945 non-null  str  
 7   order_source    646945 non-null  str  
dtypes: int64(3), str(5)
memory usage: 39.5 MB


In [5]:
# Convert order_date to datetime type
order_df["order_date"] = pd.to_datetime(order_df["order_date"]) 

# Sorting by customer_id and order_date      
order_df = order_df.sort_values(['customer_id', 'order_date'])

# Calculate inter
order_df['inter_order_gap'] = order_df.groupby('customer_id')['order_date'].diff().dt.days

# Drop null value (first orrder 
gaps = order_df['inter_order_gap'].dropna()

# 6. Tính trung vị (Median)
median_gap = gaps.median()

print(f"Trung vị số ngày giữa hai lần mua liên tiếp là: {median_gap} ngày")

Trung vị số ngày giữa hai lần mua liên tiếp là: 144.0 ngày


**Q3. Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join returns với products theo product_id), lý do trả hàng nào xuất hiện nhiều nhất?**
- A) defective
- **B) wrong_size**
- C) changed_mind
- D) not_as_described

In [6]:
return_orders = pd.read_csv(DATA_PATH / "raw" / "returns.csv")
return_orders.head()

,return_id,order_id,product_id,return_date,return_reason,return_quantity,refund_amount
0,RET-000001,2,609,2012-07-25,late_delivery,6,52458.01
1,RET-000002,32,1862,2012-07-16,wrong_size,2,5141.37
2,RET-000003,35,2359,2012-07-16,wrong_size,1,5315.95
3,RET-000004,47,1449,2012-07-11,wrong_size,4,6493.75
4,RET-000005,47,1450,2012-07-25,wrong_size,1,1740.76


In [7]:
products_df = pd.read_csv(DATA_PATH / "raw" / "products.csv")
products_df.head()

,product_id,product_name,category,segment,size,color,price,cogs
0,536,SaigonFlex UC-01,Streetwear,Everyday,S,green,11059.650000,9704.842875
1,537,SaigonFlex UC-02,Streetwear,Everyday,M,silver,9523.076013,5393.870254
2,538,SaigonFlex UC-03,Streetwear,Everyday,L,pink,15951.633158,11371.919278
3,539,SaigonFlex UC-04,Streetwear,Everyday,XL,yellow,15753.717299,8573.172954
4,540,SaigonFlex UC-05,Streetwear,Everyday,S,red,15766.334536,14063.570406


In [8]:
return_orders = pd.merge(return_orders, products_df, on="product_id", how="left")
return_orders.head()

,return_id,order_id,product_id,return_date,return_reason,return_quantity,refund_amount,product_name,category,segment,size,color,price,cogs
0,RET-000001,2,609,2012-07-25,late_delivery,6,52458.01,SaigonFlex UC-74,Streetwear,Everyday,M,yellow,10426.571034,8987.704231
1,RET-000002,32,1862,2012-07-16,wrong_size,2,5141.37,SaigonCore YY-57,GenZ,Trendy,L,orange,2656.232069,1842.628186
2,RET-000003,35,2359,2012-07-16,wrong_size,1,5315.95,VietMotion UC-07,Streetwear,Everyday,XL,yellow,5399.825901,3136.758866
3,RET-000004,47,1449,2012-07-11,wrong_size,4,6493.75,VietMode RP-41,Outdoor,Activewear,M,yellow,1802.115000,1575.769356
4,RET-000005,47,1450,2012-07-25,wrong_size,1,1740.76,VietMode RP-42,Outdoor,Activewear,L,red,1802.115000,1309.056336


In [9]:
return_orders['return_reason'][return_orders['category'] == "Streetwear"].value_counts()

return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64

**Q8. Trong các đơn hàng có order_status = ’cancelled’ trong orders.csv, phương thức thanh toán nào được sử dụng nhiều nhất?**
- **A) credit_card** 
- B) cod
- C) paypal
- D) bank_transfer

In [10]:
cancelled_orders = order_df[order_df['order_status'] == 'cancelled']
cancelled_orders['payment_method'].value_counts()

payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64

**Q9. Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có tỷ lệ trả hàng cao nhất, được định nghĩa là số bản ghi trong returns chia cho số dòng trong order_items (join với products theo product_id)?**
- **A) S**
- B) M
- C) L
- D) XL

In [11]:
order_items = pd.read_csv(DATA_PATH / "raw" / "order_items.csv")
order_items.head()

/tmp/ipykernel_906/1160839263.py:1: DtypeWarning: Columns (0: promo_id_2) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items = pd.read_csv(DATA_PATH / "raw" / "order_items.csv")


,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2
0,1,2400,7,1138.22,0.0,NaN,NaN
1,2,609,7,10166.25,0.0,NaN,NaN
2,3,396,3,11220.33,0.0,NaN,NaN
3,4,635,5,10639.25,0.0,NaN,NaN
4,6,1935,1,1597.84,0.0,NaN,NaN


In [20]:
order_items = pd.merge(order_items, products_df, on="product_id", how="left")

In [21]:
order_items.head()

,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2,product_name,category,segment,size,color,price,cogs
0,1,2400,7,1138.22,0.0,NaN,NaN,VietMotion YY-09,GenZ,Trendy,S,red,1109.261061,1053.798008
1,2,609,7,10166.25,0.0,NaN,NaN,SaigonFlex UC-74,Streetwear,Everyday,M,yellow,10426.571034,8987.704231
2,3,396,3,11220.33,0.0,NaN,NaN,SaigonFlex UM-01,Streetwear,Balanced,S,green,11028.428695,10091.012256
3,4,635,5,10639.25,0.0,NaN,NaN,SaigonFlex UC-00,Streetwear,Everyday,XL,purple,10745.220588,9205.430478
4,6,1935,1,1597.84,0.0,NaN,NaN,UrbanVN RP-10,Outdoor,Activewear,XL,purple,1609.911509,1048.696357


In [22]:
order_items['size'].value_counts()

size
XL    193025
M     176428
L     173174
S     172042
Name: count, dtype: int64

In [23]:
return_orders['size'].value_counts()

size
XL    10655
M      9820
L      9741
S      9723
Name: count, dtype: int64

In [28]:
values = [
    10655/193025,
    9820/176428,
    9741/173174,
    9723/172042
]
rounded = [round(v, 3) for v in values]
print(rounded)

[0.055, 0.056, 0.056, 0.057]
